In [4]:
import os
from pathlib import Path
import json

from dotenv import load_dotenv
from openai import OpenAI

In [2]:
# Load variables from .env into environment variables.
load_dotenv()

True

In [3]:
# client.py

def generate_notes(text: str) -> str:
    """
    Send input text to the LLM and ask for a JSON-only response.
    """

    api_key = os.getenv("OPENAI_API_KEY")
    model = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

    if not api_key:
        raise ValueError("OPENAI_API_KEY is not set.")

    client = OpenAI(api_key=api_key)

    # The prompt now explicitly defines the required JSON structure.
    # This helps make the output more stable and easier to parse.
    prompt = f"""
You are a helpful assistant for note-taking.

Read the following long-form text and return ONLY valid JSON.

Required JSON format:
{{
  "title": "string",
  "summary": "string",
  "key_points": ["string", "string"],
  "action_items": ["string", "string"],
  "tags": ["string", "string"]
}}

Rules:
- Return valid JSON only.
- Do not add markdown fences.
- Do not add explanations before or after the JSON.
- "title" should be a short descriptive title for the text.
- "summary" should be concise but informative.
- "key_points" should contain 3 to 5 items.
- "action_items" should contain 2 to 4 items when applicable.
- "tags" should contain 3 to 5 short tags.

Text:
{text}
""".strip()

    response = client.responses.create(
        model=model,
        input=prompt,
    )

    return response.output_text.strip()

In [5]:
# src/utils/parser.py

REQUIRED_FIELDS = {
    "title": str,
    "summary": str,
    "key_points": list,
    "action_items": list,
    "tags": list,
}


def parse_json_output(raw_output: str) -> dict:
    """
    Parse the raw model output into a Python dictionary.

    This function assumes the model should return JSON text.
    If the JSON is invalid, json.loads will raise an exception.
    """
    return json.loads(raw_output)


def validate_notes(data: dict) -> None:
    """
    Perform basic validation on the parsed result.

    Checks:
    - required fields exist
    - each field has the expected type
    """
    for field_name, expected_type in REQUIRED_FIELDS.items():
        if field_name not in data:
            raise ValueError(f"Missing required field: {field_name}")

        if not isinstance(data[field_name], expected_type):
            raise ValueError(
                f"Field '{field_name}' must be of type {expected_type.__name__}"
            )

    # Validate that list-based fields contain strings.
    for list_field in ["key_points", "action_items", "tags"]:
        if not all(isinstance(item, str) for item in data[list_field]):
            raise ValueError(f"All items in '{list_field}' must be strings")

In [12]:
# src/main.py

def load_input_text(file_path: Path) -> str:
    """
    Read the input text file and return its content as a string.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"Input file not found: {file_path}")

    return file_path.read_text(encoding="utf-8").strip()


def print_notes(result: dict) -> None:
    """
    Print the structured result in a human-readable format.
    """
    print("=== Structured Notes ===")
    print(f"Title: {result['title']}")
    print()
    print("Summary:")
    print(result["summary"])
    print()

    print("Key Points:")
    for item in result["key_points"]:
        print(f"- {item}")
    print()

    print("Action Items:")
    for item in result["action_items"]:
        print(f"- {item}")
    print()

    print("Tags:")
    print(", ".join(result["tags"]))
    print()


def main() -> None:
    """
    1. Load input text
    2. Generate JSON output from the LLM
    3. Parse JSON into a Python dict
    4. Validate the structure
    5. Print the result
    """
    input_file = Path("../examples/input.txt")
    text = load_input_text(input_file)

    print("=== Input Loaded ===")
    print(f"Characters: {len(text)}")
    print()

    raw_output = generate_notes(text)

    print("=== Raw Model Output ===")
    print(raw_output)
    print()

    result = parse_json_output(raw_output)
    validate_notes(result)

    print_notes(result)


In [ ]:
main()